# GradLint quickstart

[GradLint](https://github.com/marcotag93/GradLint) is a fast quality-control *gate*
for diffusion MRI: it audits the gradient scheme, detects b-vector flip /
axis-permutation errors, and writes an auto-repaired gradient table with a
machine-readable provenance log — in seconds, before a pipeline runs.

This notebook runs the full workflow on a **public** dataset: the Stanford HARDI
acquisition shipped with [DIPY](https://dipy.org/). The data is downloaded and
cached automatically on first run (~90 MB to `~/.dipy`); nothing else is needed.

```bash
pip install gradlint dipy
```

In [ ]:
from pathlib import Path

import numpy as np
from dipy.data import get_fnames

import gradlint as gg

# Stanford HARDI: 150 directions at b=2000 plus 10 b=0 volumes, 2 mm isotropic.
dwi, bval, bvec = get_fnames("stanford_hardi")
print(f"gradlint {gg.__version__} (core {gg.core_version()})")
print("dwi:", dwi)

## 1. Inspect the gradient scheme

`inspect` runs scheme QC alone — no image required. It clusters the b-values into
shells and emits advisory notes (direction counts, design-matrix conditioning,
duplicate directions, b0 drift). Here it finds one b=0 shell and one b=2000 shell.

In [ ]:
scheme = gg.inspect(bvec, bval)
print("status:", scheme.status)
for shell in scheme.shells.shells:
    kind = "b0 " if shell.is_b0 else "DWI"
    print(f"  {kind} b={shell.nominal_b:.0f}  n={shell.count}")
for note in scheme.notes:
    print("note:", note)

## 2. Audit against the image

`audit` adds flip detection on top of the scheme QC. It fits the diffusion tensor
once, derives the principal-eigenvector field for all 48 signed-permutation
candidates analytically, and ranks them with a directional-stepping fiber-coherence
index. Pipeline-gating exits are **PASS** (0), **FLAG** (0), repaired **WARN** (0), and unrepaired **WARN** (3).

In [ ]:
report = gg.audit(bvec, bval, dwi=dwi)
flip = report.flip
print("status      :", report.status)
print("decision    :", flip.decision)
print("best         :", flip.best.label)
print("runner-up    :", flip.runner_up.label)
print(f"margin       : {flip.relative_margin * 100:.1f}%")
print("recommended  :", flip.recommended_label)
print("wm voxels    :", flip.n_wm_voxels)
print(f"mask mean FA : {flip.mask_mean_fa:.2f}")
for note in report.notes:
    print("note:", note)

GradLint resolves the FSL `bvec` into the image's voxel frame from the affine. The
Stanford HARDI image is stored with a positive affine determinant, for which the
FSL convention flips the x axis; interpreted this way the distributed table is
x-flipped relative to the image, so GradLint returns **FLAG** with the recommended
convention `-x+y+z` at a wide ~17.6% margin. MRtrix3 `dwigradcheck` independently
recovers the same x-flip, cross-validating the finding.

No mask was supplied, so detection used the foreground-gated FA white-matter proxy
(mean FA ~0.28, hence the low-FA advisory). The flip is still detected
confidently; a dedicated WM mask (`mask=...`) widens the margin further.

The canonical report also renders to a compact Markdown summary:

In [ ]:
print(gg.render_markdown(report))

## 3. Repair

A repair is written only on a confident **FLAG**, and always to a **new** file —
the input gradients are never modified in place. The corrected `bvec`/`bval` and a
versioned JSON provenance log (input hashes, the full 48-candidate ranking, the
applied transform, tool version, and a UTC timestamp) are written to a temporary
directory here.

In [ ]:
import tempfile

out = Path(tempfile.mkdtemp(prefix="gradlint_quickstart_"))
fixed = gg.repair(
    dwi,
    str(out / "corrected.bvec"),
    str(out / "corrected.bval"),
    bvec=bvec,
    bval=bval,
    provenance=str(out / "provenance.json"),
)
print("decision     :", fixed.flip.decision)
print("applied remap:", fixed.repair.label)
print("written      :", [Path(p).name for p in fixed.repair.outputs])
print("provenance   :", (out / "provenance.json").exists())
print("output dir   :", out)

## 4. Verify the repair

Re-auditing the corrected table closes the loop: it now reads as **PASS** at the
identity convention `+x+y+z`.

In [ ]:
verified = gg.audit(str(out / "corrected.bvec"), str(out / "corrected.bval"), dwi=dwi)
print("status:", verified.status)
print("best   :", verified.flip.best.label)
print(f"margin : {verified.flip.relative_margin * 100:.1f}%")

## 5. Controlled corruption

Starting from the verified-clean table, we inject a known y-axis flip and re-audit.
With the ground truth known, GradLint should recover its inverse, `+x-y+z`.

In [ ]:
bvecs = np.loadtxt(out / "corrected.bvec")
bvecs[1] *= -1.0  # inject a y-axis flip into the clean table
yflip = out / "yflip.bvec"
np.savetxt(yflip, bvecs, fmt="%.6f")

caught = gg.audit(str(yflip), str(out / "corrected.bval"), dwi=dwi)
print("status:", caught.status)
print("best   :", caught.flip.best.label, "(expected +x-y+z)")
print(f"margin : {caught.flip.relative_margin * 100:.1f}%")

## 6. Self-contained HTML report

`render_html` produces a standalone report — the shell histogram, b0 drift, and the
per-candidate coherence bar chart are embedded as images, with no external files.

In [ ]:
html_path = out / "report.html"
html_path.write_text(gg.render_html(report), encoding="utf-8")
print("wrote", html_path)

## Next steps

| Verdict  | Exit | Meaning                                                       |
| -------- | :--: | ------------------------------------------------------------ |
| **PASS** | 0    | Table consistent with the image; nothing written.           |
| **WARN** | 3    | Likely issue below the repair margin; unrepaired WARN blocks. |
| **FLAG** | 0    | Confident flip / permutation; `repair` writes the fix.       |

The same operations are available on the command line
(`gradlint audit --bvec ... --bval ... --dwi ...`, `gradlint <command> --help`) and
over a whole BIDS dataset (`gradlint audit --bids DIR`). See the
[README](../README.md) and the project page at
<https://github.com/marcotag93/GradLint> for full documentation.